# Week 7 — Merging DataFrames: The Full Pipeline
## Phase 2a Python | PORA Academy Cohort 7

Yesterday you learned how `pd.merge()` joins two tables together. Today we go the
whole way: chaining **five** Olist tables into a single analysis-ready DataFrame,
bringing in review scores and payment behaviour, and answering real business
questions that no single table could answer alone.

By the end of this session, you will be able to:
- Chain multiple merges cleanly
- Use `pd.concat()` to stack DataFrames
- Handle mismatched columns after concat
- Build a complete multi-table analysis

## The Business Question

In the Olist dataset, the sales numbers live in `order_items` (112,650 rows), the
customer's home state lives in `customers`, the product category lives in
`products`, the English category name lives in a tiny `translation` table, and the
happiness of each buyer lives in `reviews`. None of these tables can answer
*"which product categories earn the most money **and** get the best reviews?"* on
their own.

To answer it, we need to **chain merges** — snapping the tables together one join
at a time until every row of `order_items` carries its customer, its category, and
its review score. Once assembled, the full table has shape **(112650, 20)** and
becomes the foundation for category revenue, state-level analysis, and payment
insights.

## Setup — Load the Olist Data

Run this cell first. It mounts Google Drive and loads all eight tables we will
merge together today. Every later cell depends on the variables created here.

In [ ]:
import pandas as pd
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define paths
olist_path = '/content/drive/MyDrive/olist-data'

# Load all datasets
orders = pd.read_csv(os.path.join(olist_path, 'olist_orders_dataset.csv'))
customers = pd.read_csv(os.path.join(olist_path, 'olist_customers_dataset.csv'))
order_items = pd.read_csv(os.path.join(olist_path, 'olist_order_items_dataset.csv'))
products = pd.read_csv(os.path.join(olist_path, 'olist_products_dataset.csv'))
reviews = pd.read_csv(os.path.join(olist_path, 'olist_order_reviews_dataset.csv'))
payments = pd.read_csv(os.path.join(olist_path, 'olist_order_payments_dataset.csv'))
sellers = pd.read_csv(os.path.join(olist_path, 'olist_sellers_dataset.csv'))
product_translations = pd.read_csv(os.path.join(olist_path, 'product_category_name_translation.csv'))

# Verify datasets loaded
print(f"✓ Loaded {len(orders):,} orders")        # expected: 99,441 orders
print(f"✓ Loaded {len(customers):,} customers")  # expected: 99,441 customers
print(f"✓ Loaded {len(order_items):,} order items")  # expected: 112,650 order items


## Concept 1 — Chaining Merges Into One Table

A **chained merge** is just several `.merge()` calls placed back-to-back, each one
feeding its result into the next. Think of it like an assembly line: the order rolls
in, we bolt on the customer's details, then the item details, then the product
category, then the English translation — and a fully-built record rolls out the end.

Because `.merge()` returns a DataFrame, we can keep calling `.merge()` on that result.
Wrapping the whole chain in parentheses lets us write one join per line so the
pipeline reads top-to-bottom like a recipe. We use `how='left'` on the product and
translation joins so that **every** order item survives even when its category is
unknown — an inner join would silently drop those rows.

In [ ]:
# Chain five tables into one analysis-ready DataFrame
full = (
    orders
    .merge(customers, on='customer_id')
    .merge(order_items, on='order_id')
    .merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
    .merge(product_translations, on='product_category_name', how='left')
)

print(full.shape)   # expected: (112650, 20)


### Now bring in the reviews

With the backbone assembled, one more **left join** attaches each order's
`review_score`. We keep it a left join so items without a review still count toward
revenue. Then a single `groupby` gives us revenue, average review, and order count
per category — the answer to our opening question.

In [ ]:
# Left join keeps every item even when no review was left
full_reviews = full.merge(
    reviews[['order_id', 'review_score']],
    on='order_id',
    how='left'
)

cat_analysis = (
    full_reviews
    .groupby('product_category_name_english')
    .agg(
        total_revenue=('price', 'sum'),
        avg_review=('review_score', 'mean'),
        order_count=('order_id', 'nunique')
    )
    .reset_index()
    .sort_values('total_revenue', ascending=False)
)

print(cat_analysis[['product_category_name_english', 'total_revenue']].head(5).round(2))
# expected top 5 by total_revenue:
# health_beauty            1258681.34
# watches_gifts            1205005.68
# bed_bath_table           1036988.68
# sports_leisure            988048.97
# computers_accessories     911954.32


## Concept 2 — Payment Behaviour

The `payments` table records *how* every order was paid: `credit_card`, `boleto`
(a Brazilian bank slip), `voucher`, or `debit_card`. Grouping by `payment_type`
tells the business which methods dominate and how many instalments customers
stretch their payments across — vital for cash-flow planning.

We aggregate three things at once (`count`, `total_value`, `avg_installments`),
then add a `pct` column so each method's share of all transactions is obvious at a
glance.

In [ ]:
# How do Olist customers pay?
pay_summary = (
    payments
    .groupby('payment_type')
    .agg(
        count=('order_id', 'count'),
        total_value=('payment_value', 'sum'),
        avg_installments=('payment_installments', 'mean')
    )
    .reset_index()
)

pay_summary['pct'] = pay_summary['count'] / pay_summary['count'].sum() * 100
pay_summary = pay_summary.sort_values('count', ascending=False)

print(pay_summary.round(2))
# expected (ordered by count):
# credit_card  76795   73.9%   avg 3.0 installments
# boleto       19784   19.0%
# voucher       5775    5.6%
# debit_card    1529    1.5%


## Using AI Assistance (DeepSeek)

From Week 4 onward you may use **DeepSeek** to help write and debug merge code —
but treat it as a junior teammate whose work you always check. The protocol:

1. **Understand first** — describe in plain English what you want *before* prompting.
2. **Prompt with context** — give the DataFrame name, the key column(s), and the exact question.
3. **Validate the output** — compare every result against the verified value in the curriculum.
4. **Never copy blindly** — you must be able to explain every line it produces.

**Sample Week 7 prompt:**

> I have two pandas DataFrames, `orders` and `customers`, that share a `customer_id`
> column. Merge them on `customer_id`, then show the merged shape and the first 3 rows.

If DeepSeek's answer disagrees with the curriculum's verified value, **the code is
wrong — not the curriculum**. Here we validate its suggestion against the known
shape `(99441, 12)`.

In [ ]:
# DeepSeek suggested this merge — we VALIDATE before trusting it:
oc = orders.merge(customers, on='customer_id')

print(oc.shape)   # expected: (99441, 12)  <- matches the curriculum, so we trust it
print(oc[['order_id', 'customer_state']].head(3))


## Going Deeper — Stacking With `pd.concat()`

Merging joins tables **side by side** (adding columns). `pd.concat()` does the
opposite: it stacks tables **on top of each other** (adding rows). Imagine three
monthly sales files with identical columns — `concat` glues them into one long
table.

The gotcha: if the pieces don't share the exact same columns, pandas keeps the
**union** of all columns and fills the missing cells with `NaN`. Knowing this
prevents nasty surprises when data arrives in slightly different shapes.

In [ ]:
# concat stacks rows vertically (small invented demo data)
q1 = pd.DataFrame({'order_id': ['A1', 'A2'], 'price': [58.90, 13.29]})
q2 = pd.DataFrame({'order_id': ['B1', 'B2'], 'price': [120.00, 45.50]})

stacked = pd.concat([q1, q2], ignore_index=True)
print(stacked.shape)   # expected: (4, 2)

# Mismatched columns -> pandas keeps the union and fills gaps with NaN
q3 = pd.DataFrame({'order_id': ['C1'], 'freight': [9.99]})
mixed = pd.concat([q1, q3], ignore_index=True)

print(mixed.columns.tolist())      # expected: ['order_id', 'price', 'freight']
print(int(mixed.isnull().sum().sum()))  # expected: 3  (two missing freight, one missing price)


## Common Mistakes

The two mistakes that bite everyone when merging: forgetting that the **default
join is inner** (so unmatched rows vanish silently), and colliding column names
that get renamed to `_x` / `_y`. Read the comments, then run the corrected code.

In [ ]:
# ── COMMON MISTAKE ───────────────────────
# WRONG - the default merge is INNER, so items with an unknown product are dropped:
# items_cat = order_items.merge(products, on='product_id')   # silently loses rows

# CORRECT - use how='left' to KEEP every item, unmatched category becomes NaN:
items_cat = order_items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)
print(items_cat['product_category_name'].isnull().sum())   # expected: 1603

# ANOTHER TRAP - if both tables share a non-key column (e.g. a 'price' column),
# pandas renames them to price_x / price_y. Select only the columns you need
# BEFORE merging (as we did with products[[...]]) to keep the result clean.


## Mini-Challenge ⏱ ~5 minutes

The `full` table (shape `(112650, 20)`) is already built above. Working on your own:

**Task:** What is the total revenue (sum of `price`) from customers in **SP** state?

**Expected output:** `5202955.05`

Fill in the blanks below.

In [ ]:
# ⏱ ~5 minutes
# The `full` DataFrame is already built above - do not rebuild it.

# TASK: total revenue (sum of 'price') for customers in SP state
# Expected: 5202955.05

# Your code here
# sp_revenue = full[full['customer_state'] == '___']['___'].___()
# print(round(sp_revenue, 2))


## Group Exercise ⏱ ~40 minutes

**Part 3 — Group Exercise**

Build the complete multi-table analysis as a team:

1. **Merge** `orders + customers + items + products + translation + reviews`.
2. **GroupBy** `customer_state` and calculate:
   - `total_revenue` (sum of `price`)
   - `avg_review_score` (mean of `review_score`)
   - `order_count` (nunique of `order_id`)
3. Which state has the **highest avg review score** among those with **>500 orders**?
4. **DeepSeek task:** Ask DeepSeek to add the `payments` table to the analysis and
   show **avg `payment_installments` by state**.
   Validate against the Brazil-wide average: **2.85 installments**.

All tables are already loaded from the Setup cell. Do not change their values.

In [ ]:
# Part 3 - Group Exercise (work together)
# All tables (orders, customers, order_items, products, product_translations,
# reviews, payments) are already loaded. Do not change them.

# 1. Build the full merged table WITH reviews
# full_reviews = (
#     orders
#     .merge(customers, on='customer_id')
#     .merge(order_items, on='order_id')
#     .merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
#     .merge(product_translations, on='product_category_name', how='left')
#     .merge(reviews[['order_id', 'review_score']], on='order_id', how='left')
# )

# 2. GroupBy customer_state -> total_revenue, avg_review_score, order_count
# state_analysis = (
#     full_reviews
#     .groupby('customer_state')
#     .agg(
#         total_revenue=('price', 'sum'),
#         avg_review_score=('review_score', 'mean'),
#         order_count=('order_id', 'nunique')
#     )
#     .reset_index()
# )
# print(state_analysis)

# 3. Highest avg review score among states with > 500 orders
# busy_states = state_analysis[state_analysis['order_count'] > 500]
# print(busy_states.sort_values('avg_review_score', ascending=False).head(1))

# 4. DeepSeek task: add payments and show avg payment_installments by state.
#    Validate the Brazil-wide average equals 2.85:
# print(round(payments['payment_installments'].mean(), 2))   # expected: 2.85


## Session Summary

| Concept | Key Syntax | Example |
|---|---|---|
| Chained merges | `df.merge(a).merge(b).merge(c)` | Build `full` = (112650, 20) |
| Left join to keep rows | `.merge(t, on='k', how='left')` | Keep items with unknown category |
| Multi-table aggregation | `.groupby(col).agg(...)` | Revenue + avg review per category |
| Payment breakdown | `payments.groupby('payment_type')` | credit_card = 73.9% of orders |
| Stack rows | `pd.concat([a, b], ignore_index=True)` | Combine monthly files |
| Mismatched concat | union of columns, gaps -> `NaN` | Missing values filled automatically |

You can now assemble any subset of the Olist tables into a single analysis table
and answer questions that span the whole business.

---
**Coming up next week (Week 8)**: we bring every Phase 2a Python skill together —
loading, cleaning, grouping, and merging — into a complete end-to-end analysis, the
final week before the capstone.